In [ ]:
import torch
import clip
from PIL import Image
import os
import pandas as pd

# Load the CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

def load_images_from_folder(folder):
    images = []
    for filename in os.listdir(folder):
        if filename.endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(folder, filename)
            image = preprocess(Image.open(img_path)).unsqueeze(0).to(device)
            images.append((filename, image))
    return images

def encode_images(images):
    encoded_images = []
    with torch.no_grad():
        for name, image in images:
            image_features = model.encode_image(image)
            encoded_images.append((name, image_features))
    return encoded_images

def compute_similarity(encoded_images1, encoded_images2):
    similarities = {}
    for name1, feat1 in encoded_images1:
        similarities[name1] = {}
        for name2, feat2 in encoded_images2:
            similarity = torch.nn.functional.cosine_similarity(feat1, feat2, dim=-1)
            similarities[name1][name2] = similarity.item()
    return similarities

folder1 = "/home/emanmunir/detection/fastapi/Template_images"
folder2 = "/home/emanmunir/detection/fastapi/Scanned_images"

images1 = load_images_from_folder(folder1)
images2 = load_images_from_folder(folder2)

encoded_images1 = encode_images(images1)
encoded_images2 = encode_images(images2)

similarities = compute_similarity(encoded_images1, encoded_images2)

# Convert similarity results to a DataFrame for easy viewing
similarity_df = pd.DataFrame(similarities).T
print(similarity_df)


In [3]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image

def extract_annotations(doc, color='red'):
    annotations_info = []
    color_map = {'red': ([0, 0, 250], [10, 10, 255])}
    for page_number in range(len(doc)):
        page = doc[page_number]
        pix = page.get_pixmap()
        if pix.samples:
            img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            lower_color, upper_color = color_map[color] if color in color_map else ([0, 0, 0], [255, 255, 255])
            mask = cv2.inRange(img, np.array(lower_color), np.array(upper_color))
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for rect in [cv2.boundingRect(cnt) for cnt in contours]:
                x0, y0, w, h = rect
                annotations_info.append((page_number + 1, x0, y0, x0 + w, y0 + h))
    return annotations_info

def remove_border(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(contour)
        img = img[y:y+h, x:x+w]
    return img, (x, y, w, h)

def extract_images(doc, annotations_info, output_folder, remove_border_flag=False):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    output_images, new_annotations_info = [], []
    for idx, (page_number, x0, y0, x1, y1) in enumerate(annotations_info):
        page = doc.load_page(page_number - 1)
        clip = fitz.Rect(x0, y0, x1, y1)
        pix = page.get_pixmap(clip=clip)
        img = cv2.imdecode(np.frombuffer(pix.tobytes(), dtype=np.uint8), cv2.IMREAD_COLOR)
        if remove_border_flag:
            img, (bx, by, bw, bh) = remove_border(img)
            new_annotations_info.append((page_number, x0 + bx, y0 + by, x0 + bx + bw, y0 + by + bh))
        else:
            new_annotations_info.append((page_number, x0, y0, x1, y1))
        output_images.append(img)
        cv2.imwrite(os.path.join(output_folder, f"extracted_image_{idx + 1}.png"), img)
    return output_images, new_annotations_info

def adjust_annotations_for_pdf2(original_annotations, source_rect, target_rect):
    adjusted_annotations = []
    x_scale = target_rect.width / source_rect.width
    y_scale = target_rect.height / source_rect.height
    for (page_number, x0, y0, x1, y1) in original_annotations:
        adjusted_annotations.append((page_number, x0 * x_scale, y0 * y_scale, x1 * x_scale, y1 * y_scale))
    return adjusted_annotations

def compute_vgg16_similarity(img1, img2):
    model = VGG16(weights='imagenet', include_top=False)
    model = Model(inputs=model.inputs, outputs=model.layers[-1].output)

    def get_features(img):
        img = cv2.resize(img, (224, 224))
        img = image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)
        return model.predict(img).flatten()

    f1, f2 = get_features(img1), get_features(img2)
    return np.dot(f1, f2) / (np.linalg.norm(f1) * np.linalg.norm(f2))

input_path1 = '/home/emanmunir/detection/test2/pdf to image/small-pdf-boxes (1).pdf'
input_path2 = "/home/emanmunir/detection/test5/scan-processed.pdf"
doc1, doc2 = fitz.open(input_path1), fitz.open(input_path2)

annotations_info1 = extract_annotations(doc1)
output_folder1, output_folder2 = "/home/emanmunir/detection/test5/pdf1", "/home/emanmunir/detection/test5/pdf2"
output_images1, new_annotations_info1 = extract_images(doc1, annotations_info1, output_folder1, remove_border_flag=True)

source_rect, target_rect = doc1[0].rect, doc2[0].rect
adjusted_annotations_info2 = adjust_annotations_for_pdf2(annotations_info1, source_rect, target_rect)
output_images2, _ = extract_images(doc2, adjusted_annotations_info2, output_folder2, remove_border_flag=True)

similarities = [compute_vgg16_similarity(img1, img2) for img1, img2 in zip(output_images1, output_images2)]
page_similarity_info = [(info[0], sim) for info, sim in zip(annotations_info1, similarities)]

print(f"Page numbers and similarities: {page_similarity_info}")


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 12s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 249ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 296ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 413ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 231ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 357ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 353ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step
1/1 ━━━━━━━━

In [ ]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image

def extract_annotations(doc, color='red'):
    annotations_info = []
    color_map = {'red': ([0, 0, 250], [10, 10, 255])}
    for page_number in range(len(doc)):
        page = doc[page_number]
        pix = page.get_pixmap()
        if pix.samples:
            img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            lower_color, upper_color = color_map[color] if color in color_map else ([0, 0, 0], [255, 255, 255])
            mask = cv2.inRange(img, np.array(lower_color), np.array(upper_color))
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for rect in [cv2.boundingRect(cnt) for cnt in contours]:
                x0, y0, w, h = rect
                annotations_info.append((page_number + 1, x0, y0, x0 + w, y0 + h))
    return annotations_info

def remove_border(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(contour)
        img = img[y:y+h, x:x+w]
    return img, (x, y, w, h)

def extract_images(doc, annotations_info, output_folder, remove_border_flag=False):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    output_images, new_annotations_info = [], []
    for idx, (page_number, x0, y0, x1, y1) in enumerate(annotations_info):
        page = doc.load_page(page_number - 1)
        clip = fitz.Rect(x0, y0, x1, y1)
        pix = page.get_pixmap(clip=clip)
        img = cv2.imdecode(np.frombuffer(pix.tobytes(), dtype=np.uint8), cv2.IMREAD_COLOR)
        if remove_border_flag:
            img, (bx, by, bw, bh) = remove_border(img)
            new_annotations_info.append((page_number, x0 + bx, y0 + by, x0 + bx + bw, y0 + by + bh))
        else:
            new_annotations_info.append((page_number, x0, y0, x1, y1))
        output_images.append(img)
        cv2.imwrite(os.path.join(output_folder, f"extracted_image_{idx + 1}.png"), img)
    return output_images, new_annotations_info

def adjust_annotations_for_pdf2(original_annotations, source_rect, target_rect):
    adjusted_annotations = []
    x_scale = target_rect.width / source_rect.width
    y_scale = target_rect.height / source_rect.height
    for (page_number, x0, y0, x1, y1) in original_annotations:
        adjusted_annotations.append((page_number, x0 * x_scale, y0 * y_scale, x1 * x_scale, y1 * y_scale))
    return adjusted_annotations

def compute_vgg16_similarity(img1, img2):
    model = VGG16(weights='imagenet', include_top=False)
    model = Model(inputs=model.inputs, outputs=model.layers[-1].output)

    def get_features(img):
        img = cv2.resize(img, (224, 224))
        img = image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)
        return model.predict(img).flatten()

    f1, f2 = get_features(img1), get_features(img2)
    return np.dot(f1, f2) / (np.linalg.norm(f1) * np.linalg.norm(f2))

def concatenate_images(img1, img2, similarity, output_path):
    concatenated_img = np.concatenate((img1, img2), axis=1)
    height, width = concatenated_img.shape[:2]
    concatenated_img = cv2.putText(concatenated_img, f'Similarity: {similarity:.2f}', (width // 4, height - 10), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
    cv2.imwrite(output_path, concatenated_img)

input_path1 = '/home/emanmunir/detection/test2/pdf to image/small-pdf-boxes (1).pdf'
input_path2 = "/home/emanmunir/detection/test5/scan-processed.pdf"
doc1, doc2 = fitz.open(input_path1), fitz.open(input_path2)

annotations_info1 = extract_annotations(doc1)
output_folder1, output_folder2 = "/home/emanmunir/detection/test5/pdf1", "/home/emanmunir/detection/test5/pdf2"
output_images1, new_annotations_info1 = extract_images(doc1, annotations_info1, output_folder1, remove_border_flag=True)

source_rect, target_rect = doc1[0].rect, doc2[0].rect
adjusted_annotations_info2 = adjust_annotations_for_pdf2(annotations_info1, source_rect, target_rect)
output_images2, _ = extract_images(doc2, adjusted_annotations_info2, output_folder2, remove_border_flag=True)

concatenated_folder = "/home/emanmunir/detection/vgg"
if not os.path.exists(concatenated_folder):
    os.makedirs(concatenated_folder)

similarities = [compute_vgg16_similarity(img1, img2) for img1, img2 in zip(output_images1, output_images2)]
for idx, (img1, img2, similarity) in enumerate(zip(output_images1, output_images2, similarities)):
    output_path = os.path.join(concatenated_folder, f"concatenated_image_{idx + 1}.png")
    concatenate_images(img1, img2, similarity, output_path)

page_similarity_info = [(info[0], sim) for info, sim in zip(annotations_info1, similarities)]
print(f"Page numbers and similarities: {page_similarity_info}")
